# M4 — Constraint Programming com CP-SAT

**Treinamento de Otimização Aplicada — Genoa para Gradus**

## Caso: Escala de Plantão do Suporte L1 da Gradus

O time de Suporte L1 cobre **segunda a sexta, 4 turnos/dia**:

| Turno | Horário |
|---|---|
| T1 | 08–12h |
| T2 | 12–16h |
| T3 | 16–20h |
| T4 | 20–24h |

São **8 analistas**. Fechar a escala respeitando:

1. Cada turno é coberto por exatamente 1 analista
2. Ninguém faz 2 turnos no mesmo dia
3. Sem T4 (noite) → T1 (manhã) no dia seguinte (descanso)
4. Máximo 4 turnos por semana por analista
5. Indisponibilidades individuais (curso, compromisso)
6. Senior só faz T3 ou T4 (cobertura fora do horário comercial)

Em MIP, regras 2, 3 e 6 viram big-M soup. Em CP-SAT, viram 3 linhas naturais.

> ⚠️ **Versão TEMPLATE** — algumas células contêm `# TODO` para você preencher. Se ficar travado, abra a versão solução: `m4_cp_solution.ipynb` (ou o `.py` em `scripts/`). Se estiver no Colab, o link da solução está no deck.

## Setup — CP-SAT vs Gurobi MIP

**Constraint Programming (CP)** e **Mixed-Integer Programming (MIP)** são duas tecnologias diferentes:

- **CP-SAT** (Google OR-Tools) propaga restrições simbolicamente. Brilha em problemas combinatórios discretos (scheduling, rostering, packing).
- **Gurobi MIP** resolve via branch-and-cut com relaxações lineares. Brilha em otimização contínua + binárias com estrutura LP-amigável.

Neste módulo modelamos a escala de plantão **nos dois**. Spoiler: CP-SAT vence em tempo e legibilidade neste tipo de problema. Mas a versão Gurobi é útil quando a única licença disponível é a do Gurobi (cenário típico em consultoria com cliente que já investiu).

In [ ]:
!pip install -q ortools gurobipy

In [ ]:
from ortools.sat.python import cp_model
from itertools import product

ANALISTAS = ['Ana', 'Bruno', 'Carla', 'Diego', 'Eduarda', 'Fábio', 'Gabi', 'Hugo']
SENIOR    = {'Eduarda', 'Hugo'}
DIAS      = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex']
TURNOS    = ['T1', 'T2', 'T3', 'T4']

INDISP = [
    ('Ana',   'Qua', 'T1'),   # curso de inglês
    ('Bruno', 'Ter', 'T2'),   # PJ no escritório
    ('Carla', 'Qui', 'T3'),   # consulta médica
    ('Diego', 'Sex', 'T4'),   # compromisso pessoal
]

N, D, T = len(ANALISTAS), len(DIAS), len(TURNOS)
print(f'{N} analistas × {D} dias × {T} turnos = {N*D*T} variáveis booleanas')

## Modelo CP-SAT — caso simples

In [ ]:
from ortools.sat.python import cp_model
from itertools import product

def montar_modelo_simples():
    """Escala simplificada: 8 analistas, 5 dias, 4 turnos/dia (1 analista por turno)."""
    ANA, DIA, TUR = range(8), range(5), range(4)
    SENIORS = {0, 1}   # ids dos seniors

    m = cp_model.CpModel()

    # TODO: variaveis booleanas x[a,d,t] (analista a faz turno t no dia d)
    # x = {(a,d,t): m.NewBoolVar(...) for a, d, t in product(ANA, DIA, TUR)}

    # TODO: restricoes
    # (1) cobertura: exatamente 1 analista por turno (use AddExactlyOne)
    # (2) max 1 turno por dia por analista (use AddAtMostOne)
    # (3) sem T4 -> T1 no dia seguinte (m.Add(x[a,d,3] + x[a,d+1,0] <= 1))
    # (4) seniors so fazem T3/T4 (forçar x[a,d,0]=x[a,d,1]=0 para a em SENIORS)

    # TODO: solver + Solve
    # s = cp_model.CpSolver(); s.Solve(m)

    raise NotImplementedError("Complete o modelo CP-SAT simples")


---

## O mesmo problema em Gurobi (MIP)

Modelagem MIP do mesmo problema. Mesmas variáveis ($x_{a,d,t}$ binárias), mas as restrições "AddExactlyOne" e "AddAtMostOne" do CP-SAT viram desigualdades algébricas:

| CP-SAT | MIP equivalente |
|---|---|
| `AddExactlyOne([v1,v2,v3])` | `v1 + v2 + v3 == 1` |
| `AddAtMostOne([v1,v2,v3])` | `v1 + v2 + v3 <= 1` |
| `Add(a + b <= 1)` | igual |
| `m.Add(x).OnlyEnforceIf(y)` | precisa big-M |

Para *esse* problema, todas as restrições são lineares simples — nenhum big-M necessário. Então MIP roda bem. O ponto frágil do MIP aparece quando há restrições condicionais ("se X então Y") — aí o big-M é difícil de calibrar.

In [ ]:
import gurobipy as gp
from gurobipy import GRB
from itertools import product

def montar_modelo_simples_gurobi():
    """Mesmo modelo em Gurobi MIP. Sem primitivos como AddExactlyOne — somas lineares."""
    m = gp.Model('escala')
    m.Params.OutputFlag = 0

    # TODO: variaveis binarias com m.addVars
    # TODO: restricoes (use x.sum('*', d, t) etc em vez de AddExactlyOne)
    # TODO: senior, descanso, etc.
    # TODO: m.optimize()

    raise NotImplementedError("Complete o modelo Gurobi MIP equivalente")


### Comentário

Solução em <50 ms. Mas observe: **sem objetivo**, o solver empilha turnos em poucos analistas. Eduarda (senior) pode acabar com 0 turnos se ninguém forçar carga mínima.

Vamos atacar isso na extensão.

---

## Exercício de extensão: 7 dias + fairness + preferências

Ampliação:
- Adicionar sábado e domingo com **2 turnos** cada (T2 12–16h, T3 16–20h) — total 24 turnos/sem
- **Fairness:** minimizar $\max_a \text{carga}_a - \min_a \text{carga}_a$
- **Preferências:** cada analista lista 2 turnos preferidos; maximizar atendidas

Combinamos os dois objetivos em uma soma ponderada lexicográfica:
$$\min\ 100 \cdot \text{gap} - \text{prefs atendidas}$$

(O peso 100 garante que fairness vem primeiro; prefs entra como desempate.)

In [ ]:
DIAS_EXT = ['Seg','Ter','Qua','Qui','Sex','Sab','Dom']

# Extensao: 7 dias. Sab e Dom tem so 2 turnos cada (T2 e T3).
# Adicione 2 objetivos:
#   1. Fairness: minimizar max_a carga_a - min_a carga_a
#   2. Preferencias: maximizar sum das preferencias atendidas
# Resolva como soma ponderada lexicografica (fairness primeiro, prefs depois).

# TODO: redefina o universo (ANA x DIA x TUR variavel por dia)
# TODO: defina variaveis carga_a (int) e gap (int) para fairness
# TODO: AddMaxEquality / AddMinEquality para implementar max-min
# TODO: defina prefs[a] = lista de 2 turnos preferidos e variavel binaria pref_ok
# TODO: resolva 2 vezes: primeiro minimizando gap; depois fixando gap e maximizando prefs

raise NotImplementedError("Complete a extensao 7 dias + fairness + preferencias")


### Para discutir

1. **Quanto tempo CP-SAT levou?** Compare com sua intuição de quanto MIP levaria.
2. **A solução é "justa"?** Cargas equilibradas (gap=0) e prefs atendidas (16/16) — é coincidência ou estrutura do problema?
3. **TODO — Comparação com MILP:** reescreva o mesmo problema no `pywraplp` (LinearSolver) e compare:
   - Número de variáveis e restrições
   - Tempo de solução
   - Legibilidade do código

4. **TODO — Stress test:** dobrar o número de analistas (16) ou adicionar 4 semanas (28 dias). CP-SAT ainda resolve em <1s?

In [ ]:
# Reescreva a escala em MILP puro (pywraplp ou Gurobi) e compare:
#   - tamanho do codigo
#   - tempo de solve
#   - facilidade de adicionar AllDifferent / NoOverlap

# TODO: implementacao MILP

raise NotImplementedError("Complete o modelo MILP equivalente para comparar com CP-SAT")


---

## Quando CP-SAT, quando Gurobi MIP? — achado contra-intuitivo

Olhe os tempos acima — em **nosso problema pequeno** (8 analistas × 20 turnos = 160 binárias, restrições todas lineares), **Gurobi venceu CP-SAT em ordem de magnitude**. Por quê?

- Gurobi tem **presolve fortíssimo** (remove redundâncias antes de procurar)
- Relaxação LP é boa nesse problema (não tem big-M)
- Branch-and-cut + cortes automáticos resolvem em poucos nós
- CP-SAT carrega overhead de propagação que só compensa em escala

### A regrinha de bolso (calibrada):

| Situação | Ferramenta recomendada |
|---|---|
| **Problemas pequenos/médios com restrições lineares**, mesmo combinatórios | **Gurobi MIP** |
| Scheduling com **NoOverlap, AllDifferent, Cumulative** em escala | **CP-SAT** |
| **Variáveis intervalares** (tempo de início/fim/duração) | **CP-SAT** |
| Lot sizing, blending, network com fluxo contínuo | **Gurobi MIP** |
| Restrições verdadeiramente lógicas ("se X então Y, exceto se Z") | **CP-SAT** |
| Cliente já tem licença Gurobi e o problema é solúvel em MIP razoável | **Gurobi** |

### Conclusão honesta para a Gradus

Para muito do que aparece em consultoria brasileira de pequeno e médio porte, **Gurobi é melhor mesmo em problemas combinatórios** — só não é se o problema explode para milhares de tarefas com NoOverlap, Cumulative, etc.

**Discurso a usar com o cliente:** "a Genoa traz Gurobi porque cobre 90% dos casos. Para o resto (scheduling industrial massivo), existem ferramentas especializadas — e é nossa obrigação como consultoria recomendar a certa."